# Testing Vec Database save

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, logging, AutoModel
logging.set_verbosity_error()
import numpy as np
import os
from torch import Tensor
from torch.utils.data import DataLoader
import faiss
import json
from beir.datasets.data_loader import GenericDataLoader

In [17]:
def stream_msmarco_chunks(path, chunk_size=5000):
    buffer = []
    with open(path, "r") as f:
        for line in f:
            doc = json.loads(line)
            buffer.append((doc["_id"], doc["text"]))

            if len(buffer) == chunk_size:
                yield buffer
                buffer = []

        if buffer:
            yield buffer

# # Loading Data
data_path = "/work/mbouthil/projects/research_project/RAG/datasets/msmarco/corpus.jsonl"

In [13]:
# Selecting Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Loading Passage Encoder
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
passage_encoder = AutoModel.from_pretrained(
    "/work/mbouthil/projects/research_project/RAG/model_weights/passage_encoder"
).to(device)
passage_encoder.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [14]:
# # Custom MSMARCO class
# class MSMARCO:
#     def __init__(self, passages):
    
#         '''
#         DataLoader for Vector Database
#         '''

#         self.passages = passages
    
#     def __len__(self):
#         return len(self.passages)
    
#     def __getitem__(self, idx):
#         passage = self.passages[idx]
#         return {"passage": passage}

In [15]:
# dataset = MSMARCO(passages)

In [18]:
# def collate_fn(batch:int, tokenizer:object, max_length:int=128) -> dict:
# 
#     passages = [x['passage'] for x in batch]
#     p_tok = tokenizer(passages, 
#                       padding=True,
#                       truncation=True, 
#                       max_length=max_length,
#                       return_tensors='pt'
#                       )
#     return {'passages': p_tok}

In [ ]:
# dataloader = DataLoader(dataset,
#                         batch_size=32,
#                         shuffle=False,
#                         num_workers=4,
#                         pin_memory=True)

In [ ]:
DIM = 768
NLIST = 4096
M = 64
NBITS = 8

TRAIN_SIZE = 100_000 
BATCH_SIZE = 64
CHUNK_SIZE = 5000

quantizer = faiss.IndexFlatIP(DIM)
index = faiss.IndexIVFPQ(
    quantizer,
    DIM,
    NLIST,
    M,
    NBITS
)

train_buf = []
train_count = 0
train_ids = []

meta_file = open("/work/mbouthil/projects/research_project/RAG/retrieval_data/passage_metadata.jsonl", "w")
global_idx = 0

for chunk in stream_msmarco_chunks(data_path, 5000):

    for i in range(0, len(chunk), BATCH_SIZE):
        batch = chunk[i : i + BATCH_SIZE]
        passages = [x[1] for x in batch]
        doc_ids = [x[0] for x in batch]

    
        with torch.no_grad():
            inputs = tokenizer(
                passages,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors='pt'
            ).to(device)

            emb = passage_encoder(**inputs).last_hidden_state[:, 0]

        emb = emb.float().cpu().numpy()
        emb = np.ascontiguousarray(emb, dtype=np.float32)
        faiss.normalize_L2(emb)

        if not index.is_trained:
            train_buf.append(emb)
            train_ids.extend(doc_ids)
            train_count += emb.shape[0]

            if train_count >= TRAIN_SIZE:
                print(f"Training FAISS index on {train_count} vectors")

                train_vecs = np.vstack(train_buf)
                index.train(train_vecs)
                index.add(train_vecs)

                # WRITE METADATA FOR TRAINING VECTORS
                for doc_id in train_ids:
                    meta_file.write(json.dumps({
                        "idx": global_idx,
                        "doc_id": doc_id
                    }) + "\n")
                    global_idx += 1

                del train_vecs
                del train_buf
                del train_ids
                train_buf = None
                train_ids = None

                print("FAISS index trained")

            continue

        if index.is_trained:
            index.add(emb)

        for doc_id in doc_ids:
            meta_file.write(json.dumps({
                "idx": global_idx,
                "doc_id": doc_id
            }) + "\n")
            global_idx += 1

        del emb, inputs
        
    torch.cuda.empty_cache()
    del chunk


meta_file.close()
faiss.write_index(index, "passage.index")

### Sanity Check

In [22]:
meta_file = open("/work/mbouthil/projects/research_project/RAG/retrieval_data/passage_metadata.jsonl", "w")

In [23]:
sum(1 for _ in open("/work/mbouthil/projects/research_project/RAG/retrieval_data/passage_metadata.jsonl"))

0

# Testing Training Loop

In [1]:
import pandas as pd
import numpy as np
from beir.datasets.data_loader import GenericDataLoader
import random
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn as nn
from torch import Tensor
import torch.nn.functional as F
import matplotlib.pyplot as plt

import time

# Loading Dataset
data_dir = "/work/mbouthil/projects/research_project/RAG/datasets/msmarco"
corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split="train")

/work/mbouthil/projects/envs/uwvenv/lib/python3.10/site-packages/beir/datasets/data_loader.py:8: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


  0%|          | 0/8841823 [00:00<?, ?it/s]

In [7]:
### Important Variables:
number_of_negatives = 2
training_batch_size = int(128/2)
dual_encoder_temp = 1
save_name = "_3"
epochs = 1


# # Loading Dataset
# data_dir = "/work/mbouthil/projects/research_project/RAG/datasets/msmarco"
# corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split="train")


class MSMARCO:
    def __init__(self,
                 queries:dict,
                 passages:dict, 
                 qrels:dict, 
                 num_negatives:int=number_of_negatives):

        '''Data loader for MS MARCO dataset'''

        self.queries = queries
        self.passages = passages
        self.qrels = qrels
        self.qids = list(self.qrels.keys())
        self.num_negatives = num_negatives
    
    def __len__(self):
        return len(self.qrels)
    
    def __getitem__(self, idx):
        qid = self.qids[idx]
        query = self.queries[qid]

        pos_pids = [k for k, v in self.qrels[qid].items() if v > 0]
        pos_passage = self.passages[random.choice(pos_pids)]['text']

        # Sample negatives directly from passages dict
        neg_passages = []
        available_pids = list(self.passages.keys())
        neg_candidates = [pid for pid in available_pids if pid not in pos_pids]
        neg_pids = random.sample(neg_candidates, min(self.num_negatives, len(neg_candidates)))
        neg_passages = [self.passages[pid]['text'] for pid in neg_pids]

        return {"query": query, "positive": pos_passage, 'negatives': neg_passages}
    

dataset = MSMARCO(queries, corpus, qrels)


# Tokenization
def collate_fn(batch, tokenizer, max_length=128):
    queries = [x['query'] for x in batch]
    positives = [x['positive'] for x in batch]
    negatives = []

    for x in batch:
        negatives.extend(x['negatives'])

    q_tok = tokenizer(
        queries, 
        padding="max_length",
        truncation=True,
        max_length=32,
        return_tensors='pt'
    )

    p_tok = tokenizer(
        positives + negatives,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

    return {"query": q_tok, "passages":p_tok}


tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# DataLoader
dataloader = DataLoader(
    dataset, 
    batch_size=training_batch_size,
    shuffle=True,
    num_workers=0,
    collate_fn=lambda x: collate_fn(x, tokenizer)
)

device = "cuda" if torch.cuda.is_available() else "cpu"

class DualEncoder(nn.Module):
    def __init__(self, query_model_name, passage_model_name):
        super().__init__()
        self.query_encoder = AutoModel.from_pretrained(query_model_name)
        self.passage_encoder = AutoModel.from_pretrained(passage_model_name)

    def encode_query(self, **inputs):
        out = self.query_encoder(**inputs)
        return out.last_hidden_state[:, 0]        # CLS
    
    def encode_passage(self, **inputs):
        out = self.passage_encoder(**inputs)
        return out.last_hidden_state[:, 0]        # CLS 
    

model = DualEncoder(
    query_model_name="bert-base-uncased",
    passage_model_name="bert-base-uncased"
).to(device)


def contrastive_loss(q_emb:Tensor, p_emb:Tensor, temperature:float=dual_encoder_temp) -> Tensor:

    '''
    Cross Entropy loss give that M_query < M_passage
    '''

    M = q_emb.shape[0]
    N = p_emb.shape[0]

    q_emb = F.normalize(q_emb, dim=-1)
    p_emb = F.normalize(p_emb, dim=-1)
    
    scores = torch.matmul(q_emb, p_emb.T)/temperature
    labels = torch.arange(M) * int(N/M)
    labels = labels.to(device=scores.device)

    loss = F.cross_entropy(scores, labels)
    return loss

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
train_loss = []

start = time.time()

for i in range(epochs):

    model.train()
    epoch_loss = 0

    for step, batch in enumerate(dataloader):

        q_inputs = {k: v.to(device) for k, v in batch["query"].items()}
        p_inputs = {k: v.to(device) for k, v in batch["passages"].items()}

        q_emb = model.encode_query(**q_inputs)
        p_emb = model.encode_passage(**p_inputs)

        loss = contrastive_loss(q_emb, p_emb)
        epoch_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        del q_emb, p_emb, loss  # Explicitly free memory
        torch.cuda.empty_cache()  
        end = time.time()
        break 

    avg_loss = epoch_loss/len(dataloader)
    train_loss.append(avg_loss)

print(end - start)

# # plotting Loss
# plt.figure(figsize=(12, 12))
# plt.suptitle("Bi-Encoder Training Loss")

# plt.plot(range(1, len(train_loss)+1), train_loss, label="Training Loss", linestyle="-", marker="o")

# plt.ylabel("Loss")
# plt.xlabel("Epoch")
# plt.legend()

# plt.style.use('bmh')
# plt.savefig("/work/mbouthil/projects/research_project/RAG/figures/loss_curve" + save_name + ".png", dpi=300)


# # Saving Encoder Weights
# save_dir = "/work/mbouthil/projects/research_project/RAG/model_weights"
# model.query_encoder.save_pretrained(f"{save_dir}/query_encoder" + save_name)
# model.passage_encoder.save_pretrained(f"{save_dir}/passage_encoder_2" + save_name)

# tokenizer.save_pretrained(save_dir)

173.9269299507141


In [8]:
print('Execution time of', end - start, 'seconds')
print('Execution time of', (end - start)/60, 'minutes')

Execution time of 173.9269299507141 seconds
Execution time of 2.898782165845235 minutes


In [2]:
### Important Variables:
number_of_negatives = 2
training_batch_size = 128
dual_encoder_temp = 1
save_name = "_3"
epochs = 1


# # Loading Dataset
# data_dir = "/work/mbouthil/projects/research_project/RAG/datasets/msmarco"
# corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split="train")


class MSMARCO:
    def __init__(self,
                 queries:dict,
                 passages:dict, 
                 qrels:dict, 
                 num_negatives:int=number_of_negatives):

        '''Data loader for MS MARCO dataset'''

        self.queries = queries
        self.passages = passages
        self.qrels = qrels
        self.qids = list(self.qrels.keys())
        self.num_negatives = num_negatives
    
    def __len__(self):
        return len(self.qrels)
    
    def __getitem__(self, idx):
        qid = self.qids[idx]
        query = self.queries[qid]

        pos_pids = [k for k, v in self.qrels[qid].items() if v > 0]
        pos_passage = self.passages[random.choice(pos_pids)]['text']

        # Sample negatives directly from passages dict
        neg_passages = []
        available_pids = list(self.passages.keys())
        neg_candidates = [pid for pid in available_pids if pid not in pos_pids]
        neg_pids = random.sample(neg_candidates, min(self.num_negatives, len(neg_candidates)))
        neg_passages = [self.passages[pid]['text'] for pid in neg_pids]

        return {"query": query, "positive": pos_passage, 'negatives': neg_passages}
    

dataset = MSMARCO(queries, corpus, qrels)


# Tokenization
def collate_fn(batch, tokenizer, max_length=128):
    queries = [x['query'] for x in batch]
    positives = [x['positive'] for x in batch]
    negatives = []

    for x in batch:
        negatives.extend(x['negatives'])

    q_tok = tokenizer(
        queries, 
        padding="max_length",
        truncation=True,
        max_length=32,
        return_tensors='pt'
    )

    p_tok = tokenizer(
        positives + negatives,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

    return {"query": q_tok, "passages":p_tok}


tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# DataLoader
dataloader = DataLoader(
    dataset, 
    batch_size=training_batch_size,
    shuffle=True,
    num_workers=0,
    collate_fn=lambda x: collate_fn(x, tokenizer)
)

device = "cuda" if torch.cuda.is_available() else "cpu"

class DualEncoder(nn.Module):
    def __init__(self, query_model_name, passage_model_name):
        super().__init__()
        self.query_encoder = AutoModel.from_pretrained(query_model_name)
        self.passage_encoder = AutoModel.from_pretrained(passage_model_name)

    def encode_query(self, **inputs):
        out = self.query_encoder(**inputs)
        return out.last_hidden_state[:, 0]        # CLS
    
    def encode_passage(self, **inputs):
        out = self.passage_encoder(**inputs)
        return out.last_hidden_state[:, 0]        # CLS 
    

model = DualEncoder(
    query_model_name="bert-base-uncased",
    passage_model_name="bert-base-uncased"
).to(device)


def contrastive_loss(q_emb:Tensor, p_emb:Tensor, temperature:float=dual_encoder_temp) -> Tensor:

    '''
    Cross Entropy loss give that M_query < M_passage
    '''

    M = q_emb.shape[0]
    N = p_emb.shape[0]

    q_emb = F.normalize(q_emb, dim=-1)
    p_emb = F.normalize(p_emb, dim=-1)
    
    scores = torch.matmul(q_emb, p_emb.T)/temperature
    labels = torch.arange(M) * int(N/M)
    labels = labels.to(device=scores.device)

    loss = F.cross_entropy(scores, labels)
    return loss

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
train_loss = []

start = time.time()

for i in range(epochs):

    model.train()
    epoch_loss = 0

    for step, batch in enumerate(dataloader):

        q_inputs = {k: v.to(device) for k, v in batch["query"].items()}
        p_inputs = {k: v.to(device) for k, v in batch["passages"].items()}

        q_emb = model.encode_query(**q_inputs)
        p_emb = model.encode_passage(**p_inputs)

        loss = contrastive_loss(q_emb, p_emb)
        epoch_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        del q_emb, p_emb, loss  # Explicitly free memory
        torch.cuda.empty_cache()  
        end = time.time()
        break 

    avg_loss = epoch_loss/len(dataloader)
    train_loss.append(avg_loss)

print(end - start)

# # plotting Loss
# plt.figure(figsize=(12, 12))
# plt.suptitle("Bi-Encoder Training Loss")

# plt.plot(range(1, len(train_loss)+1), train_loss, label="Training Loss", linestyle="-", marker="o")

# plt.ylabel("Loss")
# plt.xlabel("Epoch")
# plt.legend()

# plt.style.use('bmh')
# plt.savefig("/work/mbouthil/projects/research_project/RAG/figures/loss_curve" + save_name + ".png", dpi=300)


# # Saving Encoder Weights
# save_dir = "/work/mbouthil/projects/research_project/RAG/model_weights"
# model.query_encoder.save_pretrained(f"{save_dir}/query_encoder" + save_name)
# model.passage_encoder.save_pretrained(f"{save_dir}/passage_encoder_2" + save_name)

# tokenizer.save_pretrained(save_dir)

339.91543459892273


In [3]:
print('Execution time of', end - start, 'seconds')
print('Execution time of', (end - start)/60, 'minutes')

Execution time of 339.91543459892273 seconds
Execution time of 5.665257243315379 minutes


5.8

In [4]:
### Important Variables:
number_of_negatives = 2
training_batch_size = 128*2
dual_encoder_temp = 1
save_name = "_3"
epochs = 1


# # Loading Dataset
# data_dir = "/work/mbouthil/projects/research_project/RAG/datasets/msmarco"
# corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split="train")


class MSMARCO:
    def __init__(self,
                 queries:dict,
                 passages:dict, 
                 qrels:dict, 
                 num_negatives:int=number_of_negatives):

        '''Data loader for MS MARCO dataset'''

        self.queries = queries
        self.passages = passages
        self.qrels = qrels
        self.qids = list(self.qrels.keys())
        self.num_negatives = num_negatives
    
    def __len__(self):
        return len(self.qrels)
    
    def __getitem__(self, idx):
        qid = self.qids[idx]
        query = self.queries[qid]

        pos_pids = [k for k, v in self.qrels[qid].items() if v > 0]
        pos_passage = self.passages[random.choice(pos_pids)]['text']

        # Sample negatives directly from passages dict
        neg_passages = []
        available_pids = list(self.passages.keys())
        neg_candidates = [pid for pid in available_pids if pid not in pos_pids]
        neg_pids = random.sample(neg_candidates, min(self.num_negatives, len(neg_candidates)))
        neg_passages = [self.passages[pid]['text'] for pid in neg_pids]

        return {"query": query, "positive": pos_passage, 'negatives': neg_passages}
    

dataset = MSMARCO(queries, corpus, qrels)


# Tokenization
def collate_fn(batch, tokenizer, max_length=128):
    queries = [x['query'] for x in batch]
    positives = [x['positive'] for x in batch]
    negatives = []

    for x in batch:
        negatives.extend(x['negatives'])

    q_tok = tokenizer(
        queries, 
        padding="max_length",
        truncation=True,
        max_length=32,
        return_tensors='pt'
    )

    p_tok = tokenizer(
        positives + negatives,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

    return {"query": q_tok, "passages":p_tok}


tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# DataLoader
dataloader = DataLoader(
    dataset, 
    batch_size=training_batch_size,
    shuffle=True,
    num_workers=0,
    collate_fn=lambda x: collate_fn(x, tokenizer)
)

device = "cuda" if torch.cuda.is_available() else "cpu"

class DualEncoder(nn.Module):
    def __init__(self, query_model_name, passage_model_name):
        super().__init__()
        self.query_encoder = AutoModel.from_pretrained(query_model_name)
        self.passage_encoder = AutoModel.from_pretrained(passage_model_name)

    def encode_query(self, **inputs):
        out = self.query_encoder(**inputs)
        return out.last_hidden_state[:, 0]        # CLS
    
    def encode_passage(self, **inputs):
        out = self.passage_encoder(**inputs)
        return out.last_hidden_state[:, 0]        # CLS 
    

model = DualEncoder(
    query_model_name="bert-base-uncased",
    passage_model_name="bert-base-uncased"
).to(device)


def contrastive_loss(q_emb:Tensor, p_emb:Tensor, temperature:float=dual_encoder_temp) -> Tensor:

    '''
    Cross Entropy loss give that M_query < M_passage
    '''

    M = q_emb.shape[0]
    N = p_emb.shape[0]

    q_emb = F.normalize(q_emb, dim=-1)
    p_emb = F.normalize(p_emb, dim=-1)
    
    scores = torch.matmul(q_emb, p_emb.T)/temperature
    labels = torch.arange(M) * int(N/M)
    labels = labels.to(device=scores.device)

    loss = F.cross_entropy(scores, labels)
    return loss

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
train_loss = []

start = time.time()

for i in range(epochs):

    model.train()
    epoch_loss = 0

    for step, batch in enumerate(dataloader):

        q_inputs = {k: v.to(device) for k, v in batch["query"].items()}
        p_inputs = {k: v.to(device) for k, v in batch["passages"].items()}

        q_emb = model.encode_query(**q_inputs)
        p_emb = model.encode_passage(**p_inputs)

        loss = contrastive_loss(q_emb, p_emb)
        epoch_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        del q_emb, p_emb, loss  # Explicitly free memory
        torch.cuda.empty_cache()  
        end = time.time()
        break 

    avg_loss = epoch_loss/len(dataloader)
    train_loss.append(avg_loss)

print(end - start)

# # plotting Loss
# plt.figure(figsize=(12, 12))
# plt.suptitle("Bi-Encoder Training Loss")

# plt.plot(range(1, len(train_loss)+1), train_loss, label="Training Loss", linestyle="-", marker="o")

# plt.ylabel("Loss")
# plt.xlabel("Epoch")
# plt.legend()

# plt.style.use('bmh')
# plt.savefig("/work/mbouthil/projects/research_project/RAG/figures/loss_curve" + save_name + ".png", dpi=300)


# # Saving Encoder Weights
# save_dir = "/work/mbouthil/projects/research_project/RAG/model_weights"
# model.query_encoder.save_pretrained(f"{save_dir}/query_encoder" + save_name)
# model.passage_encoder.save_pretrained(f"{save_dir}/passage_encoder_2" + save_name)

# tokenizer.save_pretrained(save_dir)

669.8236107826233


In [5]:
print('Execution time of', end - start, 'seconds')
print('Execution time of', (end - start)/60, 'minutes')

Execution time of 669.8236107826233 seconds
Execution time of 11.163726846377054 minutes
